# Figure 5: what each type looks like

The types of the final model.  **Left**: $\phi_{kfw}$ for the categorical and ordinal variables (a binary variable shows its second level only; the first is the complement).  Numbers above the heatmap are the types' mean weight across records (%); numbers to the right the out-of-sample variance explained (`blat explain`: Brier $R^2$ for categorical, level $R^2$ for ordinal variables).  **Right**: the value each type expects for the continuous variables, in the units of the data (left axis) and on the model's $y$ scale (right axis), with $R^2$ on the $y$ scale.

Input: `RESULTS/final/K*-c1/` (final chain 1), `RESULTS/explain/final/`.

**Where to start Jupyter.**  Start it from the top directory (`jupyter lab`), so the file
browser shows the notebooks and the results side by side.  Starting it anywhere else
works too: Jupyter always runs a notebook in the notebook's own folder, and the first
cell finds the top directory from there.

**Which results.**  Set `RESULTS` in the first cell to the results directory of your run,
relative to the top directory: `example/results-quick` (the default) or
`example/results-thorough`.  When `run_final.sh` executes the notebooks it sets it for
you, through the environment variable `BLAT_RESULTS`.

In [ ]:
import os, sys
from pathlib import Path

# the top directory: the nearest folder, from here upwards, that holds the blat package
TOP = next(p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "blat" / "__init__.py").exists())
sys.path.insert(0, str(TOP))

# the results to draw, relative to the top directory (or absolute)
RESULTS = TOP / os.environ.get("BLAT_RESULTS", "example/results-quick")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blat import figures as F

def save(fig, name):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=300, bbox_inches="tight")
    print("wrote", FIGURES / f"{name}.pdf")

In [ ]:
final_paths = sorted(p for p in (RESULTS / "final").glob("K*-c1") if (p / "summary.yaml").exists())
final = F.Run(final_paths[0])
print(final.name, "K =", final.K, "|", final.record["draws"], "draws")

In [ ]:
explain_path = RESULTS / "explain" / "final"
explained = F.load_explained(explain_path) if explain_path.exists() else None
order = F.usage_order(final)
usage = final.theta_mean.mean(axis=0)[order]
table, owner = F.discrete_table(final, order)
table.round(3)

In [ ]:
fig = F.plot_discrete(table, owner, usage=usage, explained=explained)
save(fig, f"figure5-types-K{final.K}-discrete")

In [ ]:
cont = F.continuous_table(final, order)
cont.round(3)

In [ ]:
fig = F.plot_continuous(final, cont, explained=explained)
save(fig, f"figure5-types-K{final.K}-continuous")

Out-of-sample against in-sample (apparent) variance explained, if computed:

In [ ]:
if explain_path.exists():
    scores = F.load_scores(explain_path)
    display(scores[[c for c in ("kind", "r2", "r2_se", "r2_apparent", "s_hat") if c in scores]].round(3))